In [1]:
%%capture
!pip install transformers
!pip install tqdm
!pip install python-dotenv
!pip install accelerate
!pip install --upgrade transformers

In [6]:
import os
import glob

from datasets import load_dataset
from dotenv import load_dotenv
import torch
import tqdm
from transformers import AutoModelForImageTextToText, AutoProcessor
import json

load_dotenv()

def format_message(question: str, image, prompt: str = "Answer briefly") -> list:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": f"{prompt}: {question}"}
            ]
        }
    ]
    return messages


def load_model_and_processor(model_id: str):
    model = AutoModelForImageTextToText.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype=torch.bfloat16,
    )
    processor = AutoProcessor.from_pretrained(model_id)
    # Set padding side to left for batch generation
    processor.tokenizer.padding_side = "left"
    return model, processor


def load_data_from_disk(dataset_path: str, files_regex: str, split_name: str = "validation"):
    files = glob.glob(os.path.join(dataset_path, "**", "*.arrow"), recursive=True)
    val_files = [f for f in files if files_regex in f]
    data = load_dataset(
        "arrow",
        data_files={split_name: val_files},
        split=split_name,
    )
    return data


def load_data_from_hf(dataset_name: str = "lmms-lab/textvqa", split: str = "validation"):
    data = load_dataset(dataset_name, split=split, streaming=True)
    return data


def generate_and_decode(model, processor, inputs):
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=100, pad_token_id=processor.tokenizer.eos_token_id)
        # Trim out the prompt tokens to decode only the predictions
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )[0]
    return output_text

In [7]:
# model_id = "Qwen/Qwen3.5-0.8B"
model_id = "Qwen/Qwen3.5-2B"
LOCAL = False
prompt = "Answer briefly based on the image"
out_path = f"/content/drive/MyDrive/LLM_results/{model_id.split("/")[1]}/textvqa.json"
os.makedirs(os.path.dirname(out_path), exist_ok=True)

model, processor = load_model_and_processor(model_id)

if LOCAL:
    dataset_path = os.getenv("TEXTVQA_DATA_PATH")
    if not dataset_path:
        raise ValueError("Please set the TEXTVQA_DATA_PATH environment variable to the path of the TextVQA dataset.")
    data = load_data_from_disk(dataset_path, "validation-00000")
else:
    data = load_data_from_hf()

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
batch_size = 4
n_samples = 1000
results = []

# Convert streaming dataset to an iterator for manual slicing
data_iter = iter(data)

# Progress bar for batches
for i in tqdm.tqdm(range(0, n_samples, batch_size), desc="Processing batches"):
    batch_samples = []
    try:
        for _ in range(batch_size):
            batch_samples.append(next(data_iter))
    except StopIteration:
        break

    # Prepare inputs for the batch
    batch_questions = [s["question"] for s in batch_samples]
    batch_images = [s["image"] for s in batch_samples]

    # Format messages for each sample in batch
    batch_messages = [format_message(q, img, prompt=prompt) for q, img in zip(batch_questions, batch_images)]
    batch_texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True) for msg in batch_messages]

    # Processor handles padding across the batch automatically
    inputs = processor(
        text=batch_texts,
        images=batch_images,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    # Generate for the whole batch
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=50, pad_token_id=processor.tokenizer.eos_token_id)

        # Trim prompts and decode
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        batch_outputs = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

    # Collect results
    for j, sample in enumerate(batch_samples):
        results.append({
            "question_id": sample["question_id"],
            "question": sample["question"],
            "answers": sample["answers"],
            "predicted_answer": batch_outputs[j].strip(),
        })

# Save results
with open(out_path, "w") as f:
    json.dump(results, f, indent=4)
print(f"Done! Processed {len(results)} samples.")

Processing batches:  32%|███▏      | 80/250 [01:20<02:50,  1.00s/it]